In [17]:
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter, RecursiveJsonSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms.huggingface_hub import HuggingFaceHub
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_classic.chains import RetrievalQA, retrieval_qa
from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaLLM
from langchain_groq import ChatGroq

In [ ]:
loader = PyPDFDirectoryLoader("./census")

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

final_docs = text_splitter.split_documents(docs)

final_docs[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.5 (Windows)', 'creationdate': '2023-10-19T11:35:38-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'household income in states and metropolitan areas 2022', 'moddate': '2023-11-30T12:35:09+00:00', 'title': 'Household Income in States and Metropolitan Areas: 2022', 'trapped': '/false', 'source': 'census/acsbr-017.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='KEY DEFINITIONS\nHousehold income: Includes income of the \nhouseholder and all other people 15 years and \nolder in the household, whether or not they are \nrelated to the householder.\nMedian: The point that divides the household \nincome distribution into halves, one half with \nincome above the median and the other with \nincome below the median. The median is based \non the income distribution of all households, \nincluding those with no income.\nGini index: A summary measure of income \ninequality. The Gini index v

In [9]:
final_docs[0].page_content[:500]

'KEY DEFINITIONS\nHousehold income: Includes income of the \nhouseholder and all other people 15 years and \nolder in the household, whether or not they are \nrelated to the householder.\nMedian: The point that divides the household \nincome distribution into halves, one half with \nincome above the median and the other with \nincome below the median. The median is based \non the income distribution of all households, \nincluding those with no income.\nGini index: A summary measure of income \ninequality. Th'

In [10]:
len(final_docs)

316

In [ ]:
# Embeddings using huggungface

huggingface_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", # This is a small model that is good for testing. You can use any model from Hugging Face that supports sentence embeddings.
    # model_name="BAAI/bge-small-en-v1.5", # This is a larger model that is better for production. It is also faster than the all-MiniLM-L6-v2 model.
    model_kwargs={"device": "cpu"}, # {"device": "cuda"} if you have a GPU available
    encode_kwargs={"normalize_embeddings": True}, # This is important for cosine similarity to work properly. It normalizes the embeddings to have a length of 1.
)

In [ ]:
import numpy as np

np.array(huggingface_embeddings.embed_query(final_docs[0].page_content))

In [ ]:
print(np.array(huggingface_embeddings.embed_query(final_docs[0].page_content)).shape)

In [ ]:
vectorstore = FAISS.from_documents(final_docs, huggingface_embeddings)

query = "What is health insurance coverage?"

result = vectorstore.similarity_search(query=query, k=3)

print(result[0].page_content)

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
import os 

os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACE_ACCESS_TOKEN")

In [ ]:
huggingface_hub_llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-v0.2",
    model_kwargs={"temperature": 0, "max_length": 500},
)

query = "What is health insurance coverage?"
huggingface_hub_llm.invoke(query)

In [ ]:
huggingface_pipeline_llm = HuggingFacePipeline.from_model_id(
    model_id="mistralai/Mistral-7B-v0.2",
    pipeline_kwargs={"temperature": 0, "max_length": 300},
    task="text-generation",
)

llm = huggingface_pipeline_llm
llm.invoke(query)

In [ ]:
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

<context>
{context}
</context>

Question: {question}

Helpful Answers:
"""

In [ ]:
prompt = PromptTemplate.from_template(
    template=prompt_template,
    input_varibles=["context", "question"],
)

In [ ]:
openai_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
ollama_llm = OllamaLLM(model="langchain-ollama", temperature=0)
groq_llm = ChatGroq(model="langchain-groq", temperature=0)

qa_retrieval_chain = RetrievalQA.from_chain_type(
    llm=huggingface_pipeline_llm,
    retriever=retriever,
    chain_type="stuff",
    
)

In [ ]:
query = "Differences in the uninsured rates across different states in the US in 2022"

result = qa_retrieval_chain.invoke({"query": query})

print(result['output'])